In [1]:

import os
import numpy as np
import cv2
np.random.seed(1337)
import gc

from keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dropout, Dense, Conv2D,MaxPool2D,BatchNormalization,GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import MultiLabelBinarizer

img_size = 128 # set the image size to 128
print(os.listdir())
dataset = os.listdir("music_dataset_spectro_full_single/train") #use as labels
labels = dataset
print(labels)

['.git', '.vscode', 'acoustic_pediction_Phase2.wav', 'acoustic_prediction2.wav', 'audio_separator_dataset', 'Checkpoint.keras', 'Checkpoint_tester.py', 'Classifiaction_model_full.keras', 'Classifiaction_model_full_multi.keras', 'drum_pediction_Phase2.wav', 'drum_prediction.wav', 'large separation dataset output', 'longSongSplitter.py', 'Mix_maker.py', 'model saves', 'model_classes_full.ipynb', 'model_classes_full_multi.ipynb', 'model_classes_single.ipynb', 'model_separation(large dataset).ipynb', 'model_separation(small dataset).ipynb', 'music_dataset_spectro_full_single', 'Music_mixes_dataset_classification', 'Music_separation_dataset_large', 'music_separation_dataset_small', 'my_separator_model_prototype.keras', 'prediction images', 'README.md', 'separated_drum_part2.png', 'separated_drum_partcheck.png', 'separated_part_acoustic2.png', 'separated_part_acousticcheck.png', 'separated_part_Trumpet2.png', 'separator_model_small.keras', 'small separation dataset output', 'spectrogramMaker

In [2]:
def get_dataset_array_train(data_dir):
    data = []
    for label in labels: # loop through the labels
        print(label)
        path = os.path.join(data_dir, label)
        class_num = labels.index(label)
        for img in os.listdir(path): # loop through images 
            try:
                img_arr = cv2.imread(os.path.join(path, img),0)
                resized_arr = img_arr[:img_size,:img_size] # slice the image to be 128 128
                data.append([resized_arr, [class_num]]) # append label number and instrument spectrogram
                gc.collect() 
            except Exception as e:
                print(e)
    return np.array(data,dtype="object")

In [3]:
def get_dataset_array_test(data_dir):
    data = []
    for label in labels: # loop through the labels
        print(label)
        path = os.path.join(data_dir, label)
        class_num = labels.index(label)
        for img in os.listdir(path): # loop through images 
            try:
                img_arr = cv2.imread(os.path.join(path, img),0)
                resized_arr = img_arr[:img_size,:img_size] # slice the image to be 128 128
                data.append([resized_arr, [class_num]]) # append label number and instrument spectrogram
                gc.collect()
            except Exception as e:
                print(e)
    return np.array(data,dtype="object")

In [4]:
def get_dataset_array_valid(data_dir):
    data = []
    for label in labels: # loop through the labels
        print(label)
        path = os.path.join(data_dir, label)
        class_num = labels.index(label)
        for img in os.listdir(path):
            try:
                img_arr = cv2.imread(os.path.join(path, img),0)
                resized_arr = img_arr[:img_size,:img_size] # slice the image to be 128 128
                data.append([resized_arr, [class_num]])# append label number and instrument spectrogram
                gc.collect()
            except Exception as e:
                print(e)

    return np.array(data,dtype="object")

In [5]:
train = get_dataset_array_train("music_dataset_spectro_full_single/train/")
test = get_dataset_array_test("music_dataset_spectro_full_single/test/")
valid = get_dataset_array_valid("music_dataset_spectro_full_single/valid/")

Accordion
Acoustic_Guitar
Banjo
Bass_Guitar
Clarinet
cowbell
Dobro
Drum_set
Electric_Guitar
flute
Harmonium
Horn
Keyboard
Mandolin
Organ
Piano
Saxophone
Shakers
Tambourine
Trombone
Trumpet
Ukulele
vibraphone
Violin
Accordion
Acoustic_Guitar
Banjo
Bass_Guitar
Clarinet
cowbell
Dobro
Drum_set
Electric_Guitar
flute
Harmonium
Horn
Keyboard
Mandolin
Organ
Piano
Saxophone
Shakers
Tambourine
Trombone
Trumpet
Ukulele
vibraphone
Violin
Accordion
Acoustic_Guitar
Banjo
Bass_Guitar
Clarinet
cowbell
Dobro
Drum_set
Electric_Guitar
flute
Harmonium
Horn
Keyboard
Mandolin
Organ
Piano
Saxophone
Shakers
Tambourine
Trombone
Trumpet
Ukulele
vibraphone
Violin


In [6]:
print(train.shape) # see the shapes
print(test.shape) # see the shapes
print(valid.shape) # see the shapes

(32888, 2)
(4117, 2)
(4129, 2)


In [7]:
x_train = []
y_train = []

x_val = []
y_val = []

x_test = []
y_test = []

for feature, label in train: #loop through the train array and split into x image and y labels
    x_train.append(feature)
    y_train.append(label)

del train

for feature, label in test:#loop through the test array and split into x image and y labels
    x_test.append(feature)
    y_test.append(label)
del test

for feature, label in valid:#loop through the valid array and split into x image and y labels
    x_val.append(feature)
    y_val.append(label)
     
del valid

In [8]:
gc.collect()
x_train = np.array(x_train)/255 #normalize the images
gc.collect()
x_test = np.array(x_test)/255 #normalize the images
gc.collect()
x_val = np.array(x_val)/255 #normalize the images
gc.collect()

0

In [9]:
x_train = x_train.reshape(-1, img_size, img_size, 1) # reshape images

mlb = MultiLabelBinarizer() # to make the labels multi hot

y_train = mlb.fit_transform(y_train) # make the labels multi hot

x_val = x_val.reshape(-1, img_size, img_size, 1) # reshape images

y_val = mlb.fit_transform(y_val) # make the labels multi hot


x_test = x_test.reshape(-1, img_size, img_size, 1) # reshape images

y_test = mlb.fit_transform(y_test)  # make the labels multi hot


In [10]:
# Model setup
model = Sequential()

#first block
model.add(Conv2D(32, (3,3), activation = 'relu', padding="same", input_shape = (img_size, img_size, 1))) # input the image shape
model.add(BatchNormalization())  # normalize image
model.add(Conv2D(32, (3,3), activation = 'relu', padding="same")) 
model.add(MaxPool2D((2,2))) # downsample image

#second block
model.add(Conv2D(64, (3,3), activation = 'relu', padding="same"))
model.add(BatchNormalization()) # normalize image
model.add(Conv2D(64, (3,3), activation = 'relu', padding="same"))
model.add(MaxPool2D((2,2))) # downsample image

#third block
model.add(Conv2D(128, (3,3), activation = 'relu', padding="same"))
model.add(BatchNormalization()) # normalize image
model.add(Conv2D(128, (3,3), activation = 'relu', padding="same"))
model.add(GlobalAveragePooling2D())

#last block
model.add(Dense(units = 512, activation = 'relu')) #dense layer
model.add(Dropout(0.4))
model.add(Dense(units = 24, activation = 'sigmoid'))

model.compile(
              optimizer = 'adam', loss = 'binary_crossentropy',
              metrics = ['accuracy']
              )
     

In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │        12,312 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 365,688 (1.39 MB)

 Trainable params: 365,240 (1.39 MB)

 Non-trainable params: 448 (1.75 KB)

In [12]:
print(x_train.shape) 
print(y_train.shape)

(32888, 128, 128, 1)
(32888, 24)


In [13]:
learning_rate_reduction = ReduceLROnPlateau(monitor = 'val_accuracy', patience = 2, verbose = 1, factor = 0.3, min_lr = 0.000001)
stop_early = EarlyStopping("val_accuracy",patience = 4, verbose = 1)

In [14]:
batch_size = 32
n_epochs = 50
model.fit(x_train, y_train, batch_size = batch_size,
                    epochs = n_epochs, validation_data = (x_val, y_val),
                    callbacks = [learning_rate_reduction,stop_early], shuffle = True)

Epoch 1/50
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 453s 438ms/step - accuracy: 0.7792 - loss: 0.0523 - val_accuracy: 0.8731 - val_loss: 0.0331 - learning_rate: 0.0010
Epoch 2/50
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 451s 439ms/step - accuracy: 0.9293 - loss: 0.0180 - val_accuracy: 0.8426 - val_loss: 0.0435 - learning_rate: 0.0010
Epoch 3/50
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 0s 427ms/step - accuracy: 0.9539 - loss: 0.0123
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0003000000142492354.
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 451s 439ms/step - accuracy: 0.9573 - loss: 0.0115 - val_accuracy: 0.7731 - val_loss: 0.0694 - learning_rate: 0.0010
Epoch 4/50
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 449s 437ms/step - accuracy: 0.9822 - loss: 0.0052 - val_accuracy: 0.9688 - val_loss: 0.0088 - learning_rate: 3.0000e-04
Epoch 5/50
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 449s 437ms/step - accuracy: 0.9873 - loss: 0.0040 - val_accuracy: 0.9646 - val_loss: 0.0115 - learning_rate: 3.0000e-04
Epoch 6/50
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 449s 

In [15]:
gc.collect()
print("loss of the model is - " , model.evaluate(x_test,y_test)[0])
print("Accuracy of the model is - " , model.evaluate(x_test,y_test)[1]*100 , "%")
model.save('Classifiaction_model_single.keras')

129/129 ━━━━━━━━━━━━━━━━━━━━ 12s 95ms/step - accuracy: 0.9779 - loss: 0.0095
loss of the model is -  0.0094803711399436
129/129 ━━━━━━━━━━━━━━━━━━━━ 12s 94ms/step - accuracy: 0.9779 - loss: 0.0095
Accuracy of the model is -  97.7896511554718 %


In [16]:
predictions=model.predict(x_test)
pred_labels= np.where(predictions>0.5)

129/129 ━━━━━━━━━━━━━━━━━━━━ 13s 97ms/step


In [21]:
img = cv2.imread("music_dataset_spectro_full_single/test/Accordion/2864_Accordion.png",0)
img=img[:img_size,:img_size]
img= np.reshape(img,(-1, img_size, img_size, 1))
img=img/255

labels=np.array(labels)
pred=model.predict(img)
prediction = np.argmax(pred,1)
print(labels[prediction])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
['Accordion']


In [18]:
img = cv2.imread("Music_mixes_dataset_classification/test/mix/32836_mix.png",0)
img=img[:img_size,:img_size]
img= np.reshape(img,(-1, img_size, img_size, 1))
img=img/255

print(labels)
labels=np.array(labels)
pred=model.predict(img)
print(pred)
prediction = np.argmax(pred,1)
print(labels[prediction])

['Accordion' 'Acoustic_Guitar' 'Banjo' 'Bass_Guitar' 'Clarinet' 'cowbell'
 'Dobro' 'Drum_set' 'Electric_Guitar' 'flute' 'Harmonium' 'Horn'
 'Keyboard' 'Mandolin' 'Organ' 'Piano' 'Saxophone' 'Shakers' 'Tambourine'
 'Trombone' 'Trumpet' 'Ukulele' 'vibraphone' 'Violin']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
[[4.4486737e-07 1.0787815e-08 2.5887945e-04 1.0105626e-12 1.7102426e-10
  1.9082792e-08 5.0193199e-04 1.0893287e-08 6.6405693e-05 3.6333744e-10
  9.6681436e-09 9.8272692e-13 9.7576594e-01 4.9928184e-07 6.6038075e-10
  7.3845807e-10 4.2855509e-06 1.4404974e-07 2.1648227e-06 1.0303074e-06
  3.9774000e-06 4.4970562e-05 2.0926311e-08 1.7793427e-08]]
['Keyboard']


In [19]:
img = cv2.imread("Music_mixes_dataset_classification/test/mix/32844_mix.png",0)
img=img[:img_size,:img_size]
img= np.reshape(img,(-1, img_size, img_size, 1))
img=img/255

print(labels)
labels=np.array(labels)
pred=model.predict(img)
print(pred)
prediction = np.argmax(pred,1)
print(labels[prediction])

['Accordion' 'Acoustic_Guitar' 'Banjo' 'Bass_Guitar' 'Clarinet' 'cowbell'
 'Dobro' 'Drum_set' 'Electric_Guitar' 'flute' 'Harmonium' 'Horn'
 'Keyboard' 'Mandolin' 'Organ' 'Piano' 'Saxophone' 'Shakers' 'Tambourine'
 'Trombone' 'Trumpet' 'Ukulele' 'vibraphone' 'Violin']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
[[1.29535471e-09 2.12975239e-04 5.35822567e-11 1.54420536e-06
  2.68085856e-08 3.85817267e-10 3.07364076e-07 1.36140530e-04
  2.47681253e-02 1.23239545e-07 2.55201225e-11 1.87195166e-08
  3.00319547e-10 4.47742920e-03 1.10826575e-08 3.30536309e-09
  1.24706179e-09 9.51702759e-07 1.51558361e-05 2.49026670e-07
  3.66223105e-08 4.38519665e-09 3.94777507e-02 6.89102456e-11]]
['vibraphone']
